# 01 — Craigslist ingestion pipeline

MVP Phase 1: fetch RSS search results, parse into `VehicleListing`, upsert to PostGIS.

Run locally: `jupyter lab notebooks/` or via the `jupyter` Docker service.

In [ ]:
from notification_rake.config import settings
from notification_rake.ingestion import fetch_listings

# Craigslist RSS often 403 from Docker/datacenter IPs — auto-falls back to sample fixture
listings = fetch_listings(settings.craigslist_search_rss)
print(f"Fetched {len(listings)} listings")
listings[:3]

In [ ]:
from notification_rake.transform import geocode_listings, region_from_search_url

print(f"Search region: {region_from_search_url(settings.craigslist_search_rss)}")
geocoded = geocode_listings(listings, search_url=settings.craigslist_search_rss)
[(item.title, item.longitude, item.latitude) for item in geocoded[:3]]

## Pipeline

Full ingest → normalize → geocode → upsert → Gotify alerts:

```bash
make run CMD=pipeline
# or: python -m notification_rake pipeline
```

See `notification_rake.pipeline.run_pipeline` and `docs/functions.md`.

## Implemented in `src/notification_rake/`

| Step | Module |
|------|--------|
| Geocode (Nominatim + region centroid) | `geocode.py` |
| Upsert to PostGIS | `db.py`, `pipeline.py` |
| New-listing Gotify alerts | `alerts.py` |
| Hasura GraphQL tracking | `hasura.py`, `scripts/hasura_track.py` |